# ARCHS4 CLAMP models with MSigDB prior at 1% sample coverage (Random Sampling)

**Environment:** `clamp-analyses`

Runs CLAMPfull with the full MSigDB prior at 1% sample coverage, reusing the subsampled FBM, SVD, and CLAMPbase results already generated in `06_bp_coverage_rs/00_bp_coverage_C2CP_rs_01.ipynb`.

Steps (repeated for each seed):
1. Load existing subsampled FBM, SVD, CLAMPbase, and CLAMP_K from `c2cp_coverage_rs1_seed_*`
2. Run CLAMPfull with the MSigDB prior
3. Save results to `msigdb_coverage_rs1_seed_*`

## Load libraries

In [ ]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(Matrix)
library(here)
library(CLAMP)

source(here("config.R"))

## Configuration

In [ ]:
base_output_dir <- config$ARCHS4$DATASET_FOLDER

coverage     <- 0.01
coverage_pct <- coverage * 100

data_path <- here::here('data/archs4')

MULTIPLIER <- 100
MAX_ITER   <- 5000
n_runs     <- 3

message("Coverage: ", coverage_pct, "% — will run ", n_runs, " seeds")

## Load metadata and MSigDB prior

In [ ]:
meta          <- readRDS(file.path(base_output_dir, "metadata_filtered.rds"))
n_genes_thin  <- meta$n_genes_thin
archs4_genes  <- meta$gene_symbols_thin

msigdb_gmt    <- CLAMP:::read_gmt(file.path(data_path, "msigdb.v2026.1.Hs.symbols.gmt"))
names(msigdb_gmt) <- paste0("MSIGDB_", names(msigdb_gmt))
msigdb_pathMat <- gmtListToSparseMat(list(MSIGDB = msigdb_gmt))
msigdb_matched <- getMatchedPathwayMat(msigdb_pathMat, archs4_genes)
message("Loaded and matched MSigDB pathway matrix")

## Run CLAMPfull with MSigDB prior (3 seeds)

In [ ]:
for (run_idx in seq_len(n_runs)) {
  message("\n", strrep("=", 60))
  message("RUN ", run_idx, "/", n_runs)
  message(strrep("=", 60))

  src_dir <- file.path(base_output_dir,
    paste0("c2cp_coverage_rs", coverage_pct, "_seed_", run_idx))
  dst_dir <- file.path(base_output_dir,
    paste0("msigdb_coverage_rs", coverage_pct, "_seed_", run_idx))
  dir.create(dst_dir, showWarnings = FALSE, recursive = TRUE)

  # Load existing artifacts
  subsample_info <- readRDS(file.path(src_dir, "subsample_info.rds"))
  sample_names   <- subsample_info$sample_names
  n_samples      <- subsample_info$n_samples

  svd_result <- readRDS(file.path(src_dir, "svd.rds"))
  baseRes    <- readRDS(file.path(src_dir, "CLAMPbase.rds"))
  CLAMP_K    <- readRDS(file.path(src_dir, "CLAMP_K.rds"))

  Y_sub <- FBM(
    nrow        = n_genes_thin,
    ncol        = n_samples,
    backingfile = file.path(src_dir, "fbm_subsampled"),
    create_bk   = FALSE
  )

  message("Running CLAMPfull with MSigDB prior...")
  fullRes <- CLAMPfull(
    Y                 = Y_sub,
    svdres            = svd_result,
    priorMat          = msigdb_matched,
    clamp.base.result = baseRes,
    use_cpp           = TRUE,
    trace             = TRUE,
    multiplier        = MULTIPLIER,
    max.iter          = MAX_ITER,
    clamp_k           = CLAMP_K
  )

  fullRes$Z <- data.frame(fullRes$Z)
  rownames(fullRes$Z) <- archs4_genes
  fullRes$B <- data.frame(fullRes$B)
  colnames(fullRes$B) <- sample_names
  fullRes$summary <- fullRes$summary %>%
    dplyr::rename(LV = LV_index) %>%
    dplyr::mutate(LV = paste0('LV', LV))

  saveRDS(fullRes, file = file.path(dst_dir, "CLAMPfull_msigdb.rds"))

  model_dir <- file.path(dst_dir, "CLAMPfull_msigdb")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(fullRes$B,       file.path(model_dir, "B.csv"))
  write.csv(fullRes$Z,       file.path(model_dir, "Z.csv"))
  write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))

  rm(Y_sub, svd_result, baseRes, fullRes)
  gc()
}

message("\n", strrep("=", 60))
message("All ", n_runs, " runs completed!")
message(strrep("=", 60))